# ¿Con cuántos lanzamientos terminará China 2026?

En este cuaderno construyo un modelo para estimar cuántos lanzamientos orbitales habrá realizado China al finalizar 2026.

Utilizo los lanzamientos registrados hasta el 20 de septiembre. Antes de calcular la predicción final, comprobaré el funcionamiento del modelo simulando qué habría pronosticado en años anteriores.

El resultado incluirá una estimación central y un intervalo de incertidumbre. El objetivo no es presentar una cifra exacta, sino determinar qué resultados son más razonables a partir del ritmo histórico.

In [1]:
# importo la librería para trabajar con los datos
import pandas as pd

# recupero la tabla preparada durante el análisis exploratorio
df = pd.read_csv(
    "lanzamientos_china_limpios.csv"
)

# convierto la fecha y la hora al formato temporal
df["fecha_hora"] = pd.to_datetime(
    df["fecha_hora"],
    utc=True,
    errors="coerce"
)

# ordeno los lanzamientos cronológicamente
df = (
    df.sort_values("fecha_hora")
    .reset_index(drop=True)
)

print("dimensiones:", df.shape)

print(
    "periodo:",
    df["fecha_hora"].min(),
    "-",
    df["fecha_hora"].max()
)

display(df.tail())

dimensiones: (802, 17)
periodo: 1970-04-24 13:35:45+00:00 - 2026-09-20 04:03:00+00:00


,id,mision,fecha_hora,resultado,resultado_abreviado,operador,tipo_operador,cohete,plataforma,base_lanzamiento,pais,latitud,longitud,fecha,anio,mes,anio_mes
797,767f7827-53d8-4621-803b-f456a1e6b512,Gravity-1 | SpaceSail Polar Group #16,2026-09-15 22:00:00+00:00,Launch Successful,Success,Orienspace Technology,Commercial,Gravity-1,Yellow Sea (launch location 5),Haiyang Oriental Spaceport,China,31.200000,123.700000,2026-09-15,2026,9,2026-09
798,898f7df0-b6ce-4a5c-80b4-67e81842124e,Long March 12 | SatNet LEO Group 25,2026-09-17 00:32:00+00:00,Launch Successful,Success,China Aerospace Science and Technology Corpora...,Government,Long March 12,Commercial LC-2,"Wenchang Space Launch Site, People's Republic ...",China,19.597550,110.936481,2026-09-17,2026,9,2026-09
799,71aa4d56-0ce9-4c7d-a808-cc22f2ac5a84,Kuaizhou 11 | SpaceTY-51 & 52,2026-09-17 02:40:00+00:00,Launch Successful,Success,ExPace,Commercial,Kuaizhou 11,Launch Area 95A,"Jiuquan Satellite Launch Center, People's Repu...",China,40.969117,100.343333,2026-09-17,2026,9,2026-09
800,a527b585-f01f-4ecc-9230-711dcef72ef1,Long March 2D | PIESAT-2 13-16,2026-09-19 10:50:00+00:00,Launch Successful,Success,China Aerospace Science and Technology Corpora...,Government,Long March 2D,Launch Complex 9,"Taiyuan Satellite Launch Center, People's Repu...",China,38.863128,111.589567,2026-09-19,2026,9,2026-09
801,5af31461-bce5-4cfb-a0ee-b527cf285d90,Kinetica 1 | 9 satellites,2026-09-20 04:03:00+00:00,Launch Successful,Success,CAS Space,Commercial,Kinetica 1,Launch Area 130,"Jiuquan Satellite Launch Center, People's Repu...",China,40.818200,100.225140,2026-09-20,2026,9,2026-09


## Preparación del histórico de predicción

Para estimar cómo podría terminar 2026, observo qué ocurrió en años anteriores a partir de la misma fecha.

Calculo cuántos lanzamientos se habían realizado hasta el 20 de septiembre, cuántos se produjeron durante el resto del año y qué porcentaje del total anual estaba completado en ese momento.

In [2]:
# creo variables temporales a partir de la fecha
df["anio"] = df["fecha_hora"].dt.year
df["mes"] = df["fecha_hora"].dt.month
df["dia"] = df["fecha_hora"].dt.day

# selecciono los años completos desde 2010 hasta 2025
df_historico = df.loc[
    df["anio"].between(2010, 2025)
].copy()

# cuento todos los lanzamientos de cada año
totales_anuales = (
    df_historico.groupby("anio")
    .size()
    .rename("total_anual")
)

# selecciono los lanzamientos realizados hasta el 20 de septiembre
hasta_corte = df_historico.loc[
    (df_historico["mes"] < 9)
    |
    (
        (df_historico["mes"] == 9)
        & (df_historico["dia"] <= 20)
    )
]

# cuento los lanzamientos acumulados en la fecha de corte
acumulados_corte = (
    hasta_corte.groupby("anio")
    .size()
    .rename("hasta_20_septiembre")
)

# combino los resultados anuales
historico_prediccion = pd.concat(
    [
        totales_anuales,
        acumulados_corte
    ],
    axis=1
).reset_index()

# calculo los lanzamientos posteriores al corte
historico_prediccion["despues_del_corte"] = (
    historico_prediccion["total_anual"]
    - historico_prediccion["hasta_20_septiembre"]
)

# calculo el porcentaje completado al llegar al corte
historico_prediccion["porcentaje_completado"] = (
    historico_prediccion["hasta_20_septiembre"]
    .div(historico_prediccion["total_anual"])
    .mul(100)
    .round(1)
)

display(historico_prediccion)

,anio,total_anual,hasta_20_septiembre,despues_del_corte,porcentaje_completado
0,2010,15,8,7,53.3
1,2011,19,10,9,52.6
2,2012,19,12,7,63.2
3,2013,15,6,9,40.0
4,2014,16,5,11,31.2
5,2015,19,7,12,36.8
6,2016,22,14,8,63.6
7,2017,18,8,10,44.4
8,2018,39,25,14,64.1
9,2019,34,18,16,52.9


### El final del año tiene un peso importante

La proporción de lanzamientos completados antes del 20 de septiembre varía considerablemente durante los primeros años de la serie. Esta inestabilidad se explica, en parte, porque el número anual de misiones todavía era reducido.

Desde 2021 aparece un comportamiento mucho más estable. En los últimos cinco años completos, China había realizado entre el 60,2 % y el 64,2 % de sus lanzamientos al llegar al 20 de septiembre.

Esto significa que, recientemente, cerca de cuatro de cada diez misiones anuales se concentran después de esa fecha. Por esta razón utilizaré los años más recientes como referencia principal para 2026.

In [4]:
# selecciono los cinco años completos más recientes
referencia_reciente = (
    historico_prediccion.loc[
        historico_prediccion["anio"]
        .between(2021, 2025)
    ]
    .copy()
)

# guardo los lanzamientos observados en 2026
lanzamientos_2026 = (
    df.loc[
        df["anio"] == 2026
    ]
    .shape[0]
)

print(
    "lanzamientos observados en 2026:",
    lanzamientos_2026
)

# estimo el total usando el patrón de cada año reciente
referencia_reciente[
    "estimacion_2026"
] = (
    lanzamientos_2026
    / (
        referencia_reciente[
            "porcentaje_completado"
        ]
        / 100
    )
).round().astype(int)

# resumo las estimaciones obtenidas
print(
    "\nestimación central:",
    int(
        referencia_reciente[
            "estimacion_2026"
        ].median()
    )
)

print(
    "intervalo observado:",
    referencia_reciente[
        "estimacion_2026"
    ].min(),
    "-",
    referencia_reciente[
        "estimacion_2026"
    ].max()
)

lanzamientos observados en 2026: 68

estimación central: 110
intervalo observado: 106 - 113


### Primera estimación para 2026

China acumula 68 lanzamientos el 20 de septiembre de 2026.

Si el resto del año mantuviera la distribución temporal observada entre 2021 y 2025, el total final se situaría entre 106 y 113 lanzamientos. La estimación central es de 110.

Esta primera aproximación no intenta anticipar cada misión individual. Utiliza el porcentaje del año que normalmente se había completado en la misma fecha.

Antes de aceptar el resultado, comprobaré cómo habría funcionado este método al intentar predecir años anteriores.

In [5]:
# creo una lista para guardar las pruebas históricas
resultados_validacion = []

# simulo la predicción de cada año desde 2015
for anio_objetivo in range(2015, 2026):
    # utilizo solamente los cinco años anteriores
    referencia = historico_prediccion.loc[
        historico_prediccion["anio"].between(
            anio_objetivo - 5,
            anio_objetivo - 1
        )
    ]

    # calculo el porcentaje habitual completado
    porcentaje_referencia = (
        referencia[
            "porcentaje_completado"
        ].median()
        / 100
    )

    # recupero los datos reales del año objetivo
    fila_objetivo = (
        historico_prediccion.loc[
            historico_prediccion["anio"]
            == anio_objetivo
        ]
        .iloc[0]
    )

    # genero la predicción que habría hecho entonces
    prediccion = round(
        fila_objetivo[
            "hasta_20_septiembre"
        ]
        / porcentaje_referencia
    )

    resultados_validacion.append(
        {
            "anio": anio_objetivo,
            "real": fila_objetivo[
                "total_anual"
            ],
            "prediccion": prediccion
        }
    )

# convierto los resultados en una tabla
validacion_referencia = pd.DataFrame(
    resultados_validacion
)

# calculo los errores
validacion_referencia["error"] = (
    validacion_referencia["prediccion"]
    - validacion_referencia["real"]
)

validacion_referencia["error_absoluto"] = (
    validacion_referencia["error"].abs()
)

validacion_referencia[
    "error_porcentual_absoluto"
] = (
    validacion_referencia["error_absoluto"]
    .div(validacion_referencia["real"])
    .mul(100)
    .round(1)
)

display(validacion_referencia)

print(
    "error absoluto medio:",
    round(
        validacion_referencia[
            "error_absoluto"
        ].mean(),
        2
    ),
    "lanzamientos"
)

print(
    "error porcentual medio:",
    round(
        validacion_referencia[
            "error_porcentual_absoluto"
        ].mean(),
        2
    ),
    "%"
)

,anio,real,prediccion,error,error_absoluto,error_porcentual_absoluto
0,2015,19.0,13,-6.0,6.0,31.6
1,2016,22.0,35,13.0,13.0,59.1
2,2017,18.0,20,2.0,2.0,11.1
3,2018,39.0,62,23.0,23.0,59.0
4,2019,34.0,41,7.0,7.0,20.6
5,2020,39.0,51,12.0,12.0,30.8
6,2021,55.0,53,-2.0,2.0,3.6
7,2022,64.0,63,-1.0,1.0,1.6
8,2023,67.0,70,3.0,3.0,4.5
9,2024,68.0,70,2.0,2.0,2.9


error absoluto medio: 6.82 lanzamientos
error porcentual medio: 20.83 %


In [6]:
# selecciono el periodo comparable con la actividad actual
validacion_reciente = (
    validacion_referencia.loc[
        validacion_referencia["anio"] >= 2021
    ]
    .copy()
)

# calculo el error reciente del modelo de referencia
error_absoluto_reciente = (
    validacion_reciente[
        "error_absoluto"
    ].mean()
)

error_porcentual_reciente = (
    validacion_reciente[
        "error_porcentual_absoluto"
    ].mean()
)

sesgo_reciente = (
    validacion_reciente["error"].mean()
)

print(
    "error absoluto medio reciente:",
    round(error_absoluto_reciente, 2),
    "lanzamientos"
)

print(
    "error porcentual medio reciente:",
    round(error_porcentual_reciente, 2),
    "%"
)

print(
    "sesgo medio reciente:",
    round(sesgo_reciente, 2),
    "lanzamientos"
)

error absoluto medio reciente: 2.4 lanzamientos
error porcentual medio reciente: 3.38 %
sesgo medio reciente: -0.4 lanzamientos


### Validación de la primera estimación

El método resulta poco preciso durante los años de menor actividad, cuando unos pocos lanzamientos podían alterar mucho el porcentaje completado.

Sin embargo, su comportamiento mejora claramente desde 2021. En los últimos cinco años, las predicciones históricas se desviaron entre uno y cuatro lanzamientos del resultado real.

Por tanto, la estimación de 110 lanzamientos para 2026 constituye una referencia razonable dentro del nivel de actividad actual. Aun así, la compararé con un segundo modelo que analizará la evolución mensual.

## Modelo de series temporales

Construyo un segundo modelo que analiza la evolución mensual de los lanzamientos desde 2015.

El modelo identifica la tendencia general y los cambios estacionales. Para validarlo, simulo qué habría pronosticado el 20 de septiembre de cada año entre 2021 y 2025.

In [8]:
# importo el modelo de series temporales
from statsmodels.tsa.holtwinters import (
    ExponentialSmoothing
)

# creo una función para predecir el cierre de un año
def predecir_total_anual(anio_objetivo):
    # selecciono los datos disponibles hasta agosto
    datos_entrenamiento = df.loc[
        (
            df["fecha_hora"]
            >= "2015-01-01"
        )
        &
        (
            df["fecha_hora"]
            < f"{anio_objetivo}-09-01"
        )
    ].copy()

    # convierto los lanzamientos en una serie mensual
    serie_entrenamiento = (
        datos_entrenamiento
        .set_index("fecha_hora")
        .resample("MS")
        .size()
    )

    # entreno un modelo con tendencia y estacionalidad
    modelo_temporal = ExponentialSmoothing(
        serie_entrenamiento,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated"
    ).fit(
        optimized=True
    )

    # predigo de septiembre a diciembre
    prediccion_mensual = (
        modelo_temporal
        .forecast(4)
        .clip(lower=0)
    )

    # cuento lo observado hasta el 20 de septiembre
    observado_hasta_corte = df.loc[
        (df["anio"] == anio_objetivo)
        &
        (
            (df["mes"] < 9)
            |
            (
                (df["mes"] == 9)
                & (df["dia"] <= 20)
            )
        )
    ].shape[0]

    # estimo la parte de septiembre que todavía faltaba
    septiembre_restante = (
        prediccion_mensual.iloc[0]
        * (10 / 30)
    )

    # sumo el dato conocido y los meses pendientes
    prediccion_total = (
        observado_hasta_corte
        + septiembre_restante
        + prediccion_mensual.iloc[1:].sum()
    )

    return round(prediccion_total)


# simulo las predicciones de años anteriores
resultados_temporales = []

for anio_objetivo in range(2021, 2026):
    prediccion = predecir_total_anual(
        anio_objetivo
    )

    total_real = df.loc[
        df["anio"] == anio_objetivo
    ].shape[0]

    resultados_temporales.append(
        {
            "anio": anio_objetivo,
            "real": total_real,
            "prediccion": prediccion
        }
    )

# convierto los resultados en una tabla
validacion_temporal = pd.DataFrame(
    resultados_temporales
)

# calculo los errores del modelo
validacion_temporal["error"] = (
    validacion_temporal["prediccion"]
    - validacion_temporal["real"]
)

validacion_temporal["error_absoluto"] = (
    validacion_temporal["error"].abs()
)

validacion_temporal[
    "error_porcentual_absoluto"
] = (
    validacion_temporal["error_absoluto"]
    .div(validacion_temporal["real"])
    .mul(100)
    .round(1)
)

display(validacion_temporal)

print(
    "error absoluto medio:",
    round(
        validacion_temporal[
            "error_absoluto"
        ].mean(),
        2
    ),
    "lanzamientos"
)

print(
    "error porcentual medio:",
    round(
        validacion_temporal[
            "error_porcentual_absoluto"
        ].mean(),
        2
    ),
    "%"
)

,anio,real,prediccion,error,error_absoluto,error_porcentual_absoluto
0,2021,55,50,-5,5,9.1
1,2022,64,59,-5,5,7.8
2,2023,67,67,0,0,0.0
3,2024,68,66,-2,2,2.9
4,2025,93,81,-12,12,12.9


error absoluto medio: 4.8 lanzamientos
error porcentual medio: 6.54 %


### Comparación de los modelos

El modelo mensual reproduce correctamente 2023 y se aproxima al resultado de 2024. Sin embargo, tiene dificultades cuando el ritmo de lanzamientos aumenta con rapidez.

Su mayor error aparece en 2025, precisamente el año de mayor crecimiento. Habría pronosticado 81 lanzamientos cuando finalmente se realizaron 93.

En el periodo de validación, su error medio es de 4,8 lanzamientos, el doble que los 2,4 del modelo basado en el ritmo reciente. También muestra una tendencia a quedarse por debajo del resultado real.

Por tanto, utilizaré como modelo principal la referencia histórica reciente. Es más sencilla de explicar y, sobre todo, ha sido más precisa al predecir años que el modelo no había visto.

In [9]:
# calculo la predicción temporal para 2026
prediccion_temporal_2026 = (
    predecir_total_anual(2026)
)

# comparo los dos resultados
comparacion_modelos = pd.DataFrame(
    {
        "modelo": [
            "ritmo histórico reciente",
            "serie temporal mensual"
        ],
        "prediccion_2026": [
            110,
            prediccion_temporal_2026
        ],
        "error_validacion": [
            round(
                error_absoluto_reciente,
                2
            ),
            round(
                validacion_temporal[
                    "error_absoluto"
                ].mean(),
                2
            )
        ]
    }
)

display(comparacion_modelos)

,modelo,prediccion_2026,error_validacion
0,ritmo histórico reciente,110,2.4
1,serie temporal mensual,100,4.8


## Incertidumbre de la predicción

La cifra de 110 lanzamientos es una estimación central, no una certeza.

Para representar la incertidumbre, simulo distintos cierres de 2026 utilizando la variación observada en el porcentaje anual completado el 20 de septiembre durante los últimos cinco años.

El intervalo resultante muestra el rango que contiene el 90 % de las simulaciones. No debe interpretarse como una garantía, sino como una zona de resultados razonables si el calendario reciente continúa siendo representativo.

In [10]:
# importo la librería para realizar la simulación
import numpy as np

# recupero las proporciones observadas recientemente
proporciones_recientes = (
    referencia_reciente[
        "porcentaje_completado"
    ]
    .div(100)
)

# calculo su media y su variación
media_proporcion = (
    proporciones_recientes.mean()
)

desviacion_proporcion = (
    proporciones_recientes.std(
        ddof=1
    )
)

# fijo una semilla para poder repetir el resultado
generador = np.random.default_rng(42)

# simulo posibles proporciones para 2026
proporciones_simuladas = generador.normal(
    loc=media_proporcion,
    scale=desviacion_proporcion,
    size=10000
)

# evito valores extremos poco realistas
proporciones_simuladas = np.clip(
    proporciones_simuladas,
    0.50,
    0.75
)

# convierto las proporciones en totales anuales
totales_simulados = (
    lanzamientos_2026
    / proporciones_simuladas
)

# calculo la estimación y el intervalo del 90 %
estimacion_simulada = round(
    np.median(totales_simulados)
)

limite_inferior = round(
    np.percentile(
        totales_simulados,
        5
    )
)

limite_superior = round(
    np.percentile(
        totales_simulados,
        95
    )
)

print(
    "estimación central:",
    estimacion_simulada
)

print(
    "intervalo del 90 %:",
    limite_inferior,
    "-",
    limite_superior
)

estimación central: 110
intervalo del 90 %: 105 - 115


In [11]:
# calculo la probabilidad de superar el récord de 2025
probabilidad_superar_record = (
    (totales_simulados > 93)
    .mean()
    * 100
)

# calculo la probabilidad de alcanzar cien lanzamientos
probabilidad_superar_100 = (
    (totales_simulados >= 100)
    .mean()
    * 100
)

# calculo la probabilidad de alcanzar la estimación central
probabilidad_superar_110 = (
    (totales_simulados >= 110)
    .mean()
    * 100
)

# calculo la probabilidad de alcanzar el extremo superior
probabilidad_superar_115 = (
    (totales_simulados >= 115)
    .mean()
    * 100
)

print(
    "probabilidad de superar los 93:",
    round(probabilidad_superar_record, 1),
    "%"
)

print(
    "probabilidad de alcanzar 100:",
    round(probabilidad_superar_100, 1),
    "%"
)

print(
    "probabilidad de alcanzar 110:",
    round(probabilidad_superar_110, 1),
    "%"
)

print(
    "probabilidad de alcanzar 115:",
    round(probabilidad_superar_115, 1),
    "%"
)

probabilidad de superar los 93: 100.0 %
probabilidad de alcanzar 100: 100.0 %
probabilidad de alcanzar 110: 44.5 %
probabilidad de alcanzar 115: 3.9 %


### Qué resultados parecen más probables

Todas las simulaciones superan los 93 lanzamientos de 2025 y alcanzan al menos los 100. Dentro del comportamiento histórico reciente, ambas metas aparecen como resultados muy probables.

Alcanzar los 110 lanzamientos es más incierto: sucede en el 44,5 % de las simulaciones. Esto encaja con su papel como estimación central, alrededor de la cual se reparten los posibles resultados.

Llegar a 115 lanzamientos representa un escenario más exigente y solamente ocurre en el 3,9 % de las simulaciones.

Estos porcentajes dependen de que el calendario de final de año se comporte de forma similar al periodo 2021-2025. Retrasos, cancelaciones o cambios en los programas espaciales podrían producir un resultado diferente.

In [12]:
# importo la herramienta para personalizar el gráfico
import plotly.graph_objects as go

# preparo los valores que quiero comparar
categorias = [
    "2025 completo",
    "2026 hasta el 20 de septiembre",
    "predicción para 2026"
]

valores = [
    93,
    68,
    estimacion_simulada
]

# creo el gráfico de comparación
fig_prediccion = go.Figure()

fig_prediccion.add_trace(
    go.Bar(
        x=categorias,
        y=valores,
        marker_color=[
            "#8f99a3",
            "#f0a44b",
            "#bc2a2a"
        ],
        text=[
            "93",
            "68",
            f"{estimacion_simulada}"
        ],
        textposition="outside",
        error_y={
            "type": "data",
            "symmetric": False,
            "array": [
                0,
                0,
                limite_superior
                - estimacion_simulada
            ],
            "arrayminus": [
                0,
                0,
                estimacion_simulada
                - limite_inferior
            ],
            "color": "#24364b",
            "thickness": 2,
            "width": 8
        },
        hovertemplate=(
            "<b>%{x}</b><br>"
            "%{y} lanzamientos"
            "<extra></extra>"
        )
    )
)

# marco el récord establecido en 2025
fig_prediccion.add_hline(
    y=93,
    line_dash="dot",
    line_color="#6b7280",
    annotation_text="récord de 2025",
    annotation_position="top left"
)

fig_prediccion.update_layout(
    title=(
        "China podría cerrar 2026 "
        "con unos 110 lanzamientos"
    ),
    yaxis_title="lanzamientos orbitales",
    xaxis_title="",
    template="plotly_white",
    showlegend=False
)

fig_prediccion.show()

# guardo el gráfico interactivo
fig_prediccion.write_html(
    "prediccion_lanzamientos_china_2026.html",
    include_plotlyjs="cdn"
)

## Conclusiones de la predicción

China había completado 68 lanzamientos orbitales hasta el 20 de septiembre de 2026. Para estimar el cierre del año he comparado dos métodos diferentes.

El primero utiliza el porcentaje de actividad anual que China había completado en la misma fecha durante los cinco años anteriores. El segundo analiza la evolución mensual mediante un modelo de series temporales.

La validación histórica muestra que el método más sencillo es también el más preciso. Desde 2021, su error medio es de 2,4 lanzamientos, frente a los 4,8 del modelo temporal.

La diferencia se explica porque el modelo mensual tiene dificultades para seguir aceleraciones repentinas. En su simulación de 2025 habría pronosticado 81 lanzamientos, cuando finalmente se realizaron 93. Por ese motivo, su estimación de 100 lanzamientos para 2026 se considera demasiado conservadora.

El modelo basado en el ritmo reciente sitúa el resultado más probable en **110 lanzamientos orbitales**.

La simulación de incertidumbre establece un intervalo del 90 % entre **105 y 115 lanzamientos**. Esto significa que nueve de cada diez escenarios generados a partir del comportamiento reciente terminan dentro de ese rango.

En todas las simulaciones China supera los 93 lanzamientos de 2025 y alcanza al menos los 100. Sin embargo, este resultado no representa una garantía: indica que ambas metas son altamente probables bajo los supuestos del modelo.

Alcanzar los 110 lanzamientos ocurre en el 44,5 % de las simulaciones, mientras que llegar a 115 solo sucede en el 3,9 %. Por tanto, los 115 lanzamientos representan un escenario especialmente intenso, cercano al extremo superior de la previsión.

### Predicción final

> **China podría cerrar 2026 con aproximadamente 110 lanzamientos orbitales, dentro de un intervalo razonable de entre 105 y 115.**

Si la predicción central se cumple, China superaría el récord de 2025 en 17 lanzamientos, un crecimiento aproximado del 18,3 %.

El resultado reforzaría la conclusión del análisis exploratorio: los cuatro cohetes lanzados en 45 horas no fueron una anomalía, sino una manifestación del aumento sostenido de la capacidad espacial china.

### Limitaciones

La predicción supone que la distribución de lanzamientos durante los últimos meses de 2026 será similar a la observada recientemente.

El calendario puede cambiar por retrasos técnicos, condiciones meteorológicas, accidentes, decisiones políticas o modificaciones de última hora. Un número reducido de cancelaciones o nuevas misiones podría desplazar el resultado fuera del intervalo calculado.

Además, el modelo predice el número de lanzamientos, pero no su tamaño, coste, dificultad, carga transportada ni importancia estratégica.